In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_auc_score
import joblib

# --- 1. ŁADOWANIE DANYCH WEJŚCIOWYCH ---
# Definiujemy zestaw cech (kolumn) niezbędnych do analizy
cols_to_use = ['loan_status', 'annual_inc', 'loan_amnt', 'term', 'fico_range_low', 'dti']

print("⏳ Ładowanie danych z bazy danych...")
# Wczytujemy 200,000 wierszy (optymalizacja pod kątem pamięci RAM)
df = pd.read_csv("data/accepted_2007_to_2018Q4.csv", usecols=cols_to_use, nrows=200000)
print(f"✅ Pomyślnie załadowano {len(df)} rekordów.")

# --- 2. CZYSZCZENIE DANYCH I INŻYNIERIA CECH (Feature Engineering) ---
print("🧹 Rozpoczynanie procesu czyszczenia i transformacji danych...")

# 2.1. Filtrowanie statusów kredytowych
# Skupiamy się tylko na kredytach zakończonych (spłaconych lub odpisanych w straty)
df = df[df['loan_status'].isin(['Fully Paid', 'Charged Off'])].copy()
# Tworzenie zmiennej docelowej: 1 = Spłacony (Sukces), 0 = Niespłacony (Deficyt/Ryzyko)
df['target'] = df['loan_status'].apply(lambda x: 1 if x == 'Fully Paid' else 0)

# 2.2. Ekstrakcja okresu kredytowania (Term)
# Wyciągamy liczbę miesięcy z formatu tekstowego (np. "36 months" -> 36)
df['term'] = df['term'].str.extract('(\d+)').astype(float).fillna(36)

# 2.3. Imputacja brakujących wartości (Handling Missing Values)
df['annual_inc'] = df['annual_inc'].fillna(df['annual_inc'].mean())
df['fico_range_low'] = df['fico_range_low'].fillna(df['fico_range_low'].mean())
df = df.dropna(subset=['dti'])

# --- 2.4. FILTROWANIE WARTOŚCI ODSTAJĄCYCH (Data Cleaning) ---
# Usuwamy rekordy nierealistyczne, które mogłyby zaburzyć proces uczenia
df = df[df['annual_inc'] > 1000]    # Dochód mniejszy niż 1000/rok uznajemy za błąd
df = df[df['annual_inc'] < 300000]  # Ograniczenie wartości ekstremalnych (outliers)
df = df[df['dti'] <= 100]           # Wskaźnik DTI powyżej 100% jest statystycznie niepoprawny

# --- 2.5. TWORZENIE NOWEJ CECHY (Loan-to-Income Ratio) ---
# Stosunek kwoty kredytu do rocznego dochodu — kluczowy predyktor ryzyka.
df['loan_to_income'] = df['loan_amnt'] / df['annual_inc']

print(f"📊 Liczba rekordów w zbiorze treningowym po czyszczeniu: {len(df)}")

# --- 3. PROCES UCZENIA MODELU (Machine Learning) ---
# Definiujemy wektor cech (Features) oraz zmienną objaśnianą (Target)
feature_cols = ['annual_inc', 'loan_amnt', 'term', 'fico_range_low', 'dti', 'loan_to_income']
X = df[feature_cols]
y = df['target']

# Podział na zbiór treningowy i testowy (80/20)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("🧠 Trenowanie klasyfikatora Lasu Losowego (Random Forest)...")
# Używamy wag zbalansowanych (class_weight='balanced'), aby model radził sobie z mniejszą liczbą niespłaconych kredytów
model = RandomForestClassifier(n_estimators=100, max_depth=12, class_weight='balanced', random_state=42)
model.fit(X_train, y_train)

# --- 4. EWALUACJA MODELU (Metryki jakości) ---
y_pred = model.predict(X_test)
prob_test = model.predict_proba(X_test)[:, 1]

print("\n--- RAPORT KLASYFIKACJI ---")
print(classification_report(y_test, y_pred))
print(f"🚀 Wynik ROC-AUC Score: {roc_auc_score(y_test, prob_test):.3f}")

# --- 5. EKSPORT MODELU ---
# Zapisujemy wytrenowany model do pliku pkl w celu wdrożenia w aplikacji webowej
joblib.dump(model, "credit_model.pkl")
print("💾 Model 'credit_model.pkl' został pomyślnie zaktualizowany!")

<>:30: SyntaxWarning: invalid escape sequence '\d'
<>:30: SyntaxWarning: invalid escape sequence '\d'
C:\Users\n1107\AppData\Local\Temp\ipykernel_18060\3945326982.py:30: SyntaxWarning: invalid escape sequence '\d'
  df['term'] = df['term'].str.extract('(\d+)').astype(float).fillna(36)


⏳ Ładowanie danych z bazy danych...
✅ Pomyślnie załadowano 200000 rekordów.
🧹 Rozpoczynanie procesu czyszczenia i transformacji danych...
📊 Liczba rekordów w zbiorze treningowym po czyszczeniu: 174711
🧠 Trenowanie klasyfikatora Lasu Losowego (Random Forest)...

--- RAPORT KLASYFIKACJI ---
              precision    recall  f1-score   support

           0       0.36      0.53      0.43      7034
           1       0.87      0.76      0.81     27909

    accuracy                           0.72     34943
   macro avg       0.61      0.65      0.62     34943
weighted avg       0.76      0.72      0.73     34943

🚀 Wynik ROC-AUC Score: 0.705
💾 Model 'credit_model.pkl' został pomyślnie zaktualizowany!
